# Lab 2.2: Context Lifecycle Inspector

**Day 1 - Topic 2**

Goal: show which instruction files load at session startup and which load on demand when a file matching a rule's paths: globs is accessed.

> Open this notebook in Google Colab:
> [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kpassoubady/context-claude-companion/blob/main/day1/lab2_2_context_lifecycle_inspector/start/lab2_2_context_lifecycle_inspector.ipynb)

In [ ]:
# No extra packages needed: this lab uses the Python standard library only.
# If running locally with the course venv already activated, you can skip this cell.

In [ ]:
import re
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/kpassoubady/context-claude-companion.git"


def find_repo_root() -> Path:
    """Use the local companion checkout if the notebook runs inside it; otherwise clone it (Colab)."""
    here = Path.cwd().resolve()
    for p in [here, *here.parents]:
        if (p / "day1" / "demos" / "fixtures").is_dir():
            return p
    clone = Path("context-claude-companion").resolve()
    if not clone.exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL], check=True)
    return clone


REPO_ROOT = find_repo_root()
FIXTURES = REPO_ROOT / "day1" / "demos" / "fixtures"
print(f"Using fixtures in {FIXTURES}")


def check(label: str, ok: bool) -> None:
    """Print PASS when a result is correct, TODO while a step is unfinished."""
    print(("PASS  " if ok else "TODO  ") + label)

## Section 1: The fixture repo

`rules_repo` has a root `CLAUDE.md` and three rule files under `.claude/rules/`. Two of them declare `paths:` globs in YAML frontmatter; one does not. Look at the rule files and at one rule in full.

In [ ]:
REPO = FIXTURES / "rules_repo"

for p in sorted((REPO / ".claude" / "rules").glob("*.md")):
    print(p.relative_to(REPO))
print()
print((REPO / ".claude" / "rules" / "orders-service.md").read_text())

## Section 2: Parse a rule file

A rule file may start with a `---` frontmatter block that lists `paths:` globs. Return the globs and the body that follows.

In [ ]:
def parse_rule(text: str) -> tuple[list[str], str]:
    """Return (`paths:` globs, body) for a rule file."""
    lines = text.strip().splitlines()
    # TODO: If there are no lines, or the first line (stripped) is not "---", the rule has no frontmatter: return ([], text.strip())
    # TODO: Find end, the index of the next line after index 0 whose stripped text is "---"  (hint: next(i for i in range(1, len(lines)) if ...))
    # TODO: For each line between the two "---" lines that starts with "-", strip the dash, spaces, and quotes to get one glob; collect them in globs
    # TODO: Return (globs, body) where body is the lines after the closing "---", joined with "\n" and stripped
    return [], text.strip()

In [ ]:
globs, body = parse_rule((REPO / ".claude" / "rules" / "orders-service.md").read_text())
print(globs)
# Expected: ['services/orders/**/*.py', 'tests/orders/**/*.py']
no_globs, _ = parse_rule((REPO / ".claude" / "rules" / "global-security.md").read_text())
check("parse_rule finds both paths: globs in orders-service.md", globs == ["services/orders/**/*.py", "tests/orders/**/*.py"])
check("parse_rule returns the body without the frontmatter", body.startswith("# Orders Service Rules"))
check("A rule with no frontmatter has no globs", no_globs == [])

## Section 3: Match a path against the globs

These helpers are provided. `**/` spans directories and `*` stays inside one path segment. They load every rule in the fixture.

In [ ]:
def glob_to_regex(glob: str) -> re.Pattern:
    """`**/` spans directories, `*` stays inside one path segment."""
    out, i = "", 0
    while i < len(glob):
        if glob.startswith("**/", i):
            out += "(?:.*/)?"
            i += 3
        elif glob[i] == "*":
            out += "[^/]*"
            i += 1
        else:
            out += re.escape(glob[i])
            i += 1
    return re.compile(out + r"\Z")


def matches(path: str, globs: list[str]) -> bool:
    return any(glob_to_regex(g).match(path) for g in globs)


def load_rules() -> dict[str, tuple[list[str], str]]:
    return {
        str(p.relative_to(REPO)): parse_rule(p.read_text())
        for p in sorted((REPO / ".claude" / "rules").glob("*.md"))
    }


# The matcher is a small stand-in written for this lab, not Claude Code's own.
print(matches("services/orders/pricing.py", ["services/orders/**/*.py"]))
print(matches("scripts/legacy_migration/run.py", ["services/orders/**/*.py"]))
# Expected: True, then False

## Section 4: Resolve what loads

A rule with no `paths:` loads at startup. A rule with `paths:` loads only when an accessed file matches one of its globs. Then run three scenarios and total the guidance lines in the window.

In [ ]:
def resolve(accessed: list[str], rules: dict, root_lines: int) -> list[tuple[str, str, int]]:
    """Return (source, reason, line count) for every instruction file in the active window."""
    loaded = [("CLAUDE.md (root)", "startup", root_lines)]
    for name, (globs, body) in rules.items():
        n = len(body.splitlines())
        # TODO: If the rule has no globs, append (name, "startup (no paths: frontmatter)", n) to loaded and continue
        # TODO: Otherwise find the first accessed file f where matches(f, globs) is true
        # TODO: If there is one (call it hit), append (name, f"on-demand (matched {hit})", n)
        # Hint: hit = next((f for f in accessed if ...), None)
        pass
    return loaded

In [ ]:
rules = load_rules()
root_lines = len((REPO / "CLAUDE.md").read_text().strip().splitlines())
monolith = root_lines + sum(len(body.splitlines()) for _, body in rules.values())
print(f"Baseline: inlining every rule into one CLAUDE.md loads {monolith} lines at startup on every session.\n")

scenarios = [
    ("1. Session startup (before file reads)", []),
    ("2. Reading services/orders/pricing.py", ["services/orders/pricing.py"]),
    ("3. Control: reading scripts/legacy_migration/run.py", ["scripts/legacy_migration/run.py"]),
]
totals = []
for title, files in scenarios:
    missing = [f for f in files if not (REPO / f).exists()]
    assert not missing, f"fixture file not found: {missing}"
    print(f"=== {title} ===")
    total = 0
    for source, reason, count in resolve(files, rules, root_lines):
        total += count
        print(f"  + {source:<38} | {reason:<52} | {count:>2} lines")
    totals.append(total)
    print(f"  Guidance lines in the active window: {total} of {monolith}\n")

# Expected: 7 of 18, then 13 of 18, then 7 of 18 (with the unmodified fixture).
check("Startup loads only the root CLAUDE.md and the unscoped rule (7 lines)", len(totals) == 3 and totals[0] == 7)
check("Reading services/orders/pricing.py adds the orders rule on demand (13 lines)", len(totals) == 3 and totals[1] == 13)
check("The control file loads nothing extra (7 lines)", len(totals) == 3 and totals[2] == 7)

In [ ]:
print("Lab complete.")
print("Takeaway: The root CLAUDE.md and rules without paths: load at startup; rules with paths: load only when a matching file is accessed.")